In [0]:
import configparser
config=configparser.ConfigParser()
config.read('/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/config2.ini')

Insertion of new files

In [0]:
 %run 
 ./ingestion_utils

In [0]:
new_files = get_new_files(config["VOLUME"]["volume_path"],
config['TABLES']['processed_file_table']).collect()

In [0]:
from datetime import datetime, timezone
for file_data in new_files:
    print(file_data['file_path'])
    table_name=get_table_name(config,file_data['source_name'])
    print(table_name)
    start_time = datetime.now(timezone.utc)
    try :

      df =load_incremental_file(file_data['file_path'],table_name)
      write_bronze(df,table_name)
      update_file_tracker(file_data)
      end_time = datetime.now(timezone.utc)
      duration = (end_time - start_time).total_seconds()
      df_count=df.count()
      log_ingestion(
        file_data["source_name"],
            table_name,
            file_data["file_name"],
            file_data["file_path"],
            "processed",
            df_count,
            "no error",
            start_time,
            end_time,
            duration
      )
    except Exception as e:
      end_time = datetime.now(timezone.utc)
      duration = (end_time - start_time).total_seconds()
      log_ingestion(
            file_data["source_name"],
            table_name,
            file_data["file_name"],
            file_data["file_path"],
            "failed",
            0,
            str(e),
            start_time,
            end_time,
            duration
        )
      print(f"Error processing {file_data['file_name']}: {e}")
      


In [0]:
%sql
select * from healthcare_catalog.metadata.ingestion_log